# Fase 7: Mapa de Calor de Cobertura

Mapa de calor de cobertura/missingness (país × indicador × año) para los 5 indicadores ODS ya ingeridos, calculado sobre `raw_observations` (datos crudos), **antes** del filtro del 70% de cobertura de la Fase 2 (COVER-01) -- muestra por tanto la cobertura real previa a la exclusión, complementando la discusión MNAR de la Fase 2. Figura estática única para el anexo de la memoria (COVER-02).

In [ ]:
import sys
from pathlib import Path

# Notebook lives in notebook/, but `src/` and `data/panel.db` are relative to
# the project root -- add the project root to sys.path so this notebook runs
# correctly regardless of launch method (nbconvert, Jupyter Lab, VS Code).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from src import db, coverage


In [ ]:
# Fixed-literal SQL table names only (Security V5) -- never build the query
# string from a variable.
engine = db.get_engine(str(PROJECT_ROOT / "data" / "panel.db"))
raw_observations = pd.read_sql("SELECT * FROM raw_observations", engine)
country_reference = pd.read_sql("SELECT * FROM country_reference", engine)

print("raw_observations shape:", raw_observations.shape)
print("country_reference shape:", country_reference.shape)


In [ ]:
countries = sorted(raw_observations["country_code"].unique())
ordered_countries, boundaries = coverage.ordered_countries_with_boundaries(
    country_reference, countries
)

print(f"{len(countries)} countries, {len(boundaries)} SDG regions")


In [ ]:
# Real SDG-region distribution among the 215 ingested countries (verified live
# against data/panel.db): 8 regions, sizes 4-49 -- NOT the ~40-45/5-7 estimate
# from CONTEXT.md's pre-verification discretion note. Figure height below is
# sized around this real 215-row distribution, not the earlier estimate.
n_countries = len(ordered_countries)
fig, axes = plt.subplots(1, 5, figsize=(24, max(14, n_countries * 0.09)), sharey=True)

# Binary status palette (present/absent is a STATE, not a magnitude) --
# deliberately not a sequential cmap="Reds" ramp, which would imply a gradient
# that doesn't exist for this binary signal (RESEARCH.md Anti-Patterns).
cmap = ListedColormap(["#e8e8e8", "#2b6cb0"])  # gray = missing, blue = present

for ax, indicator in zip(axes, coverage.INDICATOR_CODES):
    matrix = coverage.build_presence_matrix(raw_observations, ordered_countries, indicator)
    sns.heatmap(matrix.astype(int), cmap=cmap, cbar=False, ax=ax, yticklabels=False)
    for b in boundaries:
        ax.axhline(y=b, color="black", linewidth=0.8)
    ax.set_title(indicator)
    ax.set_xlabel("Año")

# Per-country y-tick labels rendered only on the leftmost subplot (shared y-axis
# across all 5 panels via sharey=True). Resolves RESEARCH.md Open Question 2 /
# Assumption A3: targeting 300 DPI DIGITAL/PDF-zoom legibility for all 215
# countries, not physical print-page per-country legibility -- a print-legible
# font at this row count would require an implausibly tall figure (~17in+).
axes[0].set_yticks([i + 0.5 for i in range(n_countries)])
axes[0].set_yticklabels(ordered_countries, fontsize=3.5)

fig.legend(
    handles=[
        Patch(color="#e8e8e8", label="Sin dato"),
        Patch(color="#2b6cb0", label="Dato presente"),
    ],
    loc="lower center",
    ncol=2,
)
fig.suptitle(
    "Cobertura de datos crudos por indicador ODS, país y año (2000-2022)"
)
plt.tight_layout(rect=(0, 0.03, 1, 0.97))

# Save + show in the SAME cell as figure construction: the ipykernel inline
# backend auto-closes a figure at the end of the cell it was built in, so a
# plt.savefig() call in a LATER cell would silently create+save a new blank
# figure via plt.gcf() instead of this one. fig.savefig() (bound to the actual
# Figure object) + doing it here avoids that gotcha. Exactly ONE savefig call
# -> ONE PNG (D-02, COVER-02).
output_path = PROJECT_ROOT / "figuras" / "07_mapa_calor_cobertura.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {output_path}")


## Sanity check: cobertura sobre datos crudos (Success Criteria #2)

Verificación de que la señal de cobertura viene de `raw_observations` (datos crudos, antes del filtro del 70%), no de la tabla filtrada -- el conteo total de celdas sin dato debe reflejar el grid completo país×año×indicador crudo, no el conteo (menor) de la tabla ya filtrada.

In [ ]:
total_cells = len(ordered_countries) * len(coverage.YEARS) * len(coverage.INDICATOR_CODES)
total_present = sum(
    coverage.build_presence_matrix(raw_observations, ordered_countries, ind).to_numpy().sum()
    for ind in coverage.INDICATOR_CODES
)
total_missing = total_cells - total_present
print(f"Total cells: {total_cells}")
print(f"Present: {total_present}")
print(f"Missing (raw, pre-filter): {total_missing}")
